# Financial Fraud Detection Using GNNs - Quick Start

This notebook demonstrates the complete pipeline for fraud detection using Graph Neural Networks.

## Table of Contents
1. Data Loading
2. Preprocessing
3. Graph Construction
4. Model Training
5. Evaluation
6. Explainability

In [ ]:
# Import required libraries
import sys
sys.path.append('..')

import torch
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_preprocessing.data_loader import DataLoader
from src.data_preprocessing.preprocessor import DataPreprocessor
from src.graph_construction.graph_builder import GraphBuilder
from src.models.model_factory import create_model
from src.training.trainer import Trainer
from src.evaluation.evaluator import ModelEvaluator
from src.utils.config_loader import load_config

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Imports successful!")

## 1. Load Data

We'll use the synthetic dataset for this demo. You can change to 'ieee-cis', 'elliptic', or 'paysim' if you have those datasets.

In [ ]:
# Load synthetic dataset
data_loader = DataLoader(dataset_name='synthetic', data_dir='../data/raw')
df = data_loader.load_data()

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Visualize class distribution
fraud_counts = df['is_fraud'].value_counts()

plt.figure(figsize=(8, 6))
plt.bar(['Normal', 'Fraud'], fraud_counts.values, color=['lightblue', 'red'], alpha=0.7)
plt.ylabel('Count')
plt.title('Class Distribution')
plt.xticks(rotation=0)
for i, v in enumerate(fraud_counts.values):
    plt.text(i, v + 50, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Fraud ratio: {fraud_counts[1] / len(df) * 100:.2f}%")

## 2. Preprocess Data

Feature engineering, normalization, and train/val/test splitting.

In [ ]:
# Preprocess
preprocessor = DataPreprocessor(
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    random_seed=42
)

df_processed, metadata = preprocessor.fit_transform(df)
train_df, val_df, test_df = preprocessor.split_data(df_processed)

print(f"\nFeature columns: {metadata['feature_columns'][:5]}... ({len(metadata['feature_columns'])} total)")
print(f"\nClass imbalance info:")
for key, value in metadata['imbalance_info'].items():
    print(f"  {key}: {value}")

## 3. Build Graph

Construct transaction graph with temporal edges.

In [ ]:
# Add split column
df_processed['split'] = 'train'
df_processed.loc[val_df.index, 'split'] = 'val'
df_processed.loc[test_df.index, 'split'] = 'test'

# Build graph
graph_builder = GraphBuilder(
    temporal_edges=True,
    time_window_hours=24
)

graph_data = graph_builder.build_graph(df_processed, metadata['feature_columns'])

print(f"\nGraph created successfully!")
print(f"  Nodes: {graph_data.num_nodes:,}")
print(f"  Edges: {graph_data.num_edges:,}")
print(f"  Features per node: {graph_data.num_node_features}")
print(f"  Average degree: {graph_data.num_edges / graph_data.num_nodes:.2f}")

## 4. Train Model

We'll train the Jump-Attention GNN model.

In [ ]:
# Load config
config = load_config('../config/config.yaml')

# Create model
model = create_model(
    model_name='jump_attention',
    in_channels=graph_data.num_node_features,
    config=config['model']
)

print(f"Model: {model}")
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Train model
trainer = Trainer(
    model=model,
    data=graph_data,
    config=config['training']
)

# Train for fewer epochs in notebook
history = trainer.train(num_epochs=30)

print("\n✅ Training complete!")

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(history['train_loss'], label='Train Loss', linewidth=2)
ax1.plot(history['val_loss'], label='Val Loss', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(alpha=0.3)

# Accuracy
ax2.plot(history['train_acc'], label='Train Accuracy', linewidth=2)
ax2.plot(history['val_acc'], label='Val Accuracy', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Evaluate Model

Comprehensive evaluation on test set.

In [ ]:
# Evaluate
evaluator = ModelEvaluator(
    model=model,
    data=graph_data,
    device=str(trainer.device)
)

results = evaluator.evaluate_all_splits()

# Print test metrics
print("\n" + "="*50)
print("TEST SET PERFORMANCE")
print("="*50)
for metric, value in results['test']['metrics'].items():
    print(f"{metric:15s}: {value:.4f}")
print("="*50)

In [ ]:
# Plot confusion matrix
cm = results['test']['confusion_matrix']

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
           xticklabels=['Normal', 'Fraud'],
           yticklabels=['Normal', 'Fraud'],
           cbar_kws={'label': 'Count'})
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.title('Confusion Matrix - Test Set')
plt.tight_layout()
plt.show()

# Calculate metrics from confusion matrix
tn, fp, fn, tp = cm.ravel()
print(f"\nConfusion Matrix Breakdown:")
print(f"  True Negatives:  {tn:5d}")
print(f"  False Positives: {fp:5d}")
print(f"  False Negatives: {fn:5d}")
print(f"  True Positives:  {tp:5d}")

## 6. Explainability

Understand why the model makes predictions.

In [ ]:
# Get hop attention scores (Jump-Attention specific)
from src.explainability.attention_visualizer import AttentionVisualizer

attention_viz = AttentionVisualizer(model, graph_data)
attention_viz.visualize_hop_attention(save_path='../results/hop_attention.png')

print("✅ Hop attention visualization saved!")

In [ ]:
# Explain a fraud prediction
from src.explainability.node_importance import NodeImportanceCalculator

importance_calc = NodeImportanceCalculator(model, graph_data)

# Find a fraud case
fraud_indices = torch.where(graph_data.y == 1)[0]
fraud_idx = fraud_indices[0].item()

explanation = importance_calc.explain_prediction(fraud_idx)

print(f"\nExplanation for Node {fraud_idx}:")
print(f"  Predicted: {explanation['predicted_label']} ({explanation['predicted_probability']:.3f})")
print(f"  True Label: {explanation['true_label']}")
print(f"  Correct: {explanation['correct']}")
print(f"\n  Top 5 Important Features:")
for feat, importance in explanation['top_features']:
    print(f"    {feat}: {importance:.4f}")

## Summary

In this notebook, we:

1. ✅ Loaded and explored the synthetic fraud dataset
2. ✅ Preprocessed data with feature engineering
3. ✅ Built a transaction graph with temporal edges
4. ✅ Trained a Jump-Attention GNN model
5. ✅ Evaluated performance (ROC-AUC, F1, etc.)
6. ✅ Generated explanations for predictions

### Next Steps

- Try different models (GCN, GraphSAGE, GAT)
- Experiment with hyperparameters
- Test on real datasets (IEEE-CIS, Elliptic)
- Explore more explainability features

### For More Information

- See `README.md` for detailed documentation
- See `RESEARCH_MAPPING.md` for paper alignment
- See `VIVA_GUIDE.md` for viva preparation